# 05 - Monitoreo, deriva de esquema e incidentes

## Objetivo y preparacion

Demostrar que el gobierno continua despues de definir reglas: se necesitan metricas, alertas, responsables, decisiones y evidencia de reintento. Se utilizan Python local, pandas y el ETL SQLite definido en las celdas de este notebook. No hay conexiones cloud, tableros externos ni notificaciones reales. Cada ejecucion crea sus datos y base de pruebas.

## Paso 1 - Comparar escenarios sin cambiar el contrato

Se evaluan tres lotes: un subconjunto de 4 filas aptas, el lote original defectuoso y el subconjunto apto con un importe alterado a cero. La alteracion es intencional y ocurre solamente en una copia de laboratorio. Con 1 rechazo en 4 filas, el ultimo supera el 20% permitido y se bloquea.

Una metrica debe publicar su denominador. La tasa de rechazo usa filas distintas rechazadas; el numero de hallazgos puede ser mayor porque una fila incumple varias reglas. La cobertura de exactitud usa filas con referencia sobre total de filas; el cumplimiento de exactitud usa solo filas evaluables. Compararlos como si tuvieran el mismo denominador produce conclusiones falsas.

La tasa de rechazo es un SLI, el objetivo del contrato es un SLO y un SLA requeriria acuerdo y consecuencias. Cambiar el umbral para que un lote pase no corrige datos. El Steward investiga; el Owner autoriza excepciones o cambios; el custodio ejecuta acciones y conserva evidencia. En este notebook los roles son campos informativos, no identidades autenticadas.

### Organizacion del codigo

La siguiente celda define localmente el ETL SQLite que utiliza esta demostracion de monitoreo. Incluye la puerta, la huella, el DDL, las transacciones y la auditoria. Se reproduce aqui para que el notebook pueda ejecutarse y comprenderse sin depender de haber ejecutado el 03.

`gobierno_demo.py` aporta exclusivamente el caso de datos, las reglas de normalizacion y evaluacion y sus auxiliares comunes a los cinco notebooks. La carga y las evidencias de incidentes son visibles en este notebook; no hay un segundo modulo externo de implementacion.

In [ ]:
import hashlib
import json
import sqlite3
from uuid import uuid4

import pandas as pd


def puerta_publicacion(total, rechazadas, max_rechazo=0.20):
    if not 0 <= max_rechazo <= 1 or total < 0 or not 0 <= rechazadas <= total:
        raise ValueError('Parametros invalidos para la puerta de publicacion')
    return total > 0 and rechazadas < total and rechazadas / total <= max_rechazo


def huella(datos):
    contenido = datos.to_json(orient='split', date_format='iso', index=False)
    return hashlib.sha256(contenido.encode('utf-8')).hexdigest()


def nueva_conexion():
    conexion = sqlite3.connect(':memory:')
    conexion.execute('PRAGMA foreign_keys = ON')
    conexion.executescript('''
        CREATE TABLE productos (id_producto INTEGER PRIMARY KEY, producto TEXT NOT NULL);
        CREATE TABLE ventas (
            id_venta INTEGER PRIMARY KEY,
            id_producto INTEGER NOT NULL REFERENCES productos(id_producto),
            fecha_venta TEXT NOT NULL,
            monto_centavos INTEGER NOT NULL CHECK (monto_centavos > 0),
            moneda TEXT NOT NULL CHECK (moneda IN ('ARS', 'USD', 'EUR')),
            fila_fuente TEXT NOT NULL,
            lote_inicial TEXT NOT NULL
        );
        CREATE TABLE lotes (
            lote TEXT PRIMARY KEY, huella TEXT NOT NULL, contrato TEXT NOT NULL,
            estado TEXT NOT NULL, total INTEGER NOT NULL, aceptadas INTEGER NOT NULL,
            rechazadas INTEGER NOT NULL, nuevas INTEGER NOT NULL, existentes INTEGER NOT NULL
        );
        CREATE TABLE hallazgos (
            lote TEXT NOT NULL REFERENCES lotes(lote), fila_fuente TEXT NOT NULL,
            regla TEXT NOT NULL, accion TEXT NOT NULL
        );
    ''')
    return conexion


def ejecutar_etl(datos_crudos, productos, referencia, conexion):
    datos = demo.normalizar(datos_crudos)
    aceptadas, rechazadas, metricas, detalle = demo.evaluar(datos, productos, referencia)
    lote = uuid4().hex
    estado = 'PUBLICADO' if puerta_publicacion(len(datos), len(rechazadas)) else 'BLOQUEADO'
    nuevas, existentes = 0, 0
    try:
        with conexion:
            if estado == 'PUBLICADO':
                for producto in productos.itertuples(index=False):
                    existente = conexion.execute('SELECT producto FROM productos WHERE id_producto=?', (int(producto.id_producto),)).fetchone()
                    if existente is None:
                        conexion.execute('INSERT INTO productos VALUES (?, ?)', (int(producto.id_producto), producto.producto))
                    elif existente[0] != producto.producto:
                        raise ValueError('Cambio de maestro: requiere politica SCD aprobada')
                for venta in aceptadas.itertuples(index=False):
                    valores = (int(venta.id_producto), venta.fecha_venta.date().isoformat(), int(venta.monto_centavos), str(venta.moneda))
                    existente = conexion.execute('SELECT id_producto, fecha_venta, monto_centavos, moneda FROM ventas WHERE id_venta=?', (int(venta.id_venta),)).fetchone()
                    if existente is None:
                        conexion.execute('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?)', (int(venta.id_venta), *valores, venta.fila_fuente, lote))
                        nuevas += 1
                    elif existente == valores:
                        existentes += 1
                    else:
                        raise ValueError('Conflicto de contenido para una venta ya publicada')
                claves = [int(identificador) for identificador in aceptadas['id_venta']]
                marcadores = ','.join('?' for _ in claves)
                destino = pd.read_sql_query(f'SELECT id_venta, moneda, monto_centavos FROM ventas WHERE id_venta IN ({marcadores})', conexion, params=claves)
                assert len(destino) == len(aceptadas)
                esperado = {str(moneda): int(total) for moneda, total in aceptadas.groupby('moneda')['monto_centavos'].sum().items()}
                obtenido = {str(moneda): int(total) for moneda, total in destino.groupby('moneda')['monto_centavos'].sum().items()}
                assert esperado == obtenido
            conexion.execute('INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', (lote, huella(datos_crudos), demo.VERSION_CONTRATO, estado, len(datos), len(aceptadas), len(rechazadas), nuevas, existentes))
            conexion.executemany('INSERT INTO hallazgos VALUES (?, ?, ?, ?)', [(lote, fila.fila_fuente, fila.regla, fila.accion) for fila in detalle.itertuples(index=False)])
    except Exception:
        with conexion:
            conexion.execute('INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', (lote, huella(datos_crudos), demo.VERSION_CONTRATO, 'ERROR_TECNICO', len(datos), len(aceptadas), len(rechazadas), 0, 0))
        raise
    return {'lote': lote, 'estado': estado, 'nuevas': nuevas, 'existentes': existentes, 'aceptadas': aceptadas, 'rechazadas': rechazadas, 'metricas': metricas, 'detalle': detalle}


def json_seguro(objeto):
    return json.dumps(objeto, ensure_ascii=True, indent=2, allow_nan=False)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

candidatos = [Path.cwd(), Path.cwd() / 'notebook' / 'unidad_III']
carpeta = next((ruta for ruta in candidatos if (ruta / 'gobierno_demo.py').exists()), None)
if carpeta is None:
    raise RuntimeError('Ejecutar desde la raiz del repositorio o notebook/unidad_III')
sys.path.insert(0, str(carpeta.resolve()))
import gobierno_demo as demo

crudos, productos, referencia = demo.crear_datos()
aptas, _, _, _ = demo.evaluar(demo.normalizar(crudos), productos, referencia)
subconjunto = crudos[crudos['fila_fuente'].isin(aptas['fila_fuente'])].copy()
degradado = subconjunto.copy(deep=True)
degradado.loc[degradado.index[0], 'monto'] = '0.00'
conexion = nueva_conexion()
monitoreo = []
for nombre, fuente in [('apto', subconjunto), ('original_defectuoso', crudos), ('degradado', degradado)]:
    resultado = ejecutar_etl(fuente, productos, referencia, conexion)
    regla_exactitud = resultado['metricas'].set_index('regla').loc['DQ_EXACTITUD']
    monitoreo.append({
        'escenario': nombre,
        'lote': resultado['lote'],
        'estado': resultado['estado'],
        'total_filas': len(fuente),
        'rechazadas': len(resultado['rechazadas']),
        'tasa_rechazo': len(resultado['rechazadas']) / len(fuente),
        'hallazgos': len(resultado['detalle']),
        'cobertura_exactitud_pct': 100 * regla_exactitud['evaluados'] / len(fuente),
        'cumplimiento_exactitud_pct': regla_exactitud['cumplimiento_pct'],
        'contrato': demo.VERSION_CONTRATO,
    })
tablero = pd.DataFrame(monitoreo)
display(tablero)
assert tablero['estado'].tolist() == ['PUBLICADO', 'BLOQUEADO', 'BLOQUEADO']
assert tablero.loc[2, 'tasa_rechazo'] == 0.25
assert conexion.execute('SELECT COUNT(*) FROM ventas').fetchone()[0] == 4
print('Las ventas publicadas antes de la degradacion no se alteraron.')

## Paso 2 - Detectar deriva, abrir incidente y verificar recuperacion

Una columna perdida incumple el contrato incluso antes de evaluar filas. La funcion de normalizacion debe fallar sin publicar datos. Como el error ocurre en el preflight antes de que el modulo registre el lote, este notebook conserva un evento de orquestacion separado. En produccion debe ser persistente e incluir correlacion con el intento, alerta y responsable.

El incidente del lote degradado se representa con ID, referencia al lote, regla, severidad, asignacion, causa y accion. La causa conocida aqui es una alteracion deliberada del importe en una copia. Una recuperacion valida restaura el valor desde evidencia verificada, conserva el lote original para analisis y vuelve a ejecutar las mismas reglas. No se baja el umbral del 20%.

La autorizacion del Owner es simulada: un campo de texto no es una firma ni un sistema de workflow. El estado del incidente puede pasar a CERRADO_VERIFICADO solamente despues de una corrida apta y de comprobar el destino. Se conserva la vinculacion con el lote fallido y el lote de recuperacion.

El reintento del subconjunto original apto encuentra ventas existentes sin duplicarlas. Esto demuestra recuperacion de un escenario de prueba; no resuelve las 8 filas defectuosas del lote original de 12.

In [ ]:
eventos_orquestacion = []
try:
    ejecutar_etl(subconjunto.drop(columns=['moneda']), productos, referencia, conexion)
except ValueError as error:
    assert 'Deriva de esquema' in str(error)
    eventos_orquestacion.append({
        'tipo': 'DERIVA_ESQUEMA',
        'estado': 'BLOQUEADO_PREFLIGHT',
        'responsable': 'Steward Comercial',
        'contrato': demo.VERSION_CONTRATO,
    })
assert len(eventos_orquestacion) == 1
display(pd.DataFrame(eventos_orquestacion))

incidente = {
    'id': 'INC-DEMO-001',
    'lote_origen': tablero.loc[2, 'lote'],
    'estado': 'ASIGNADO',
    'regla': 'DQ_MONTO',
    'severidad': 'alta para publicacion del lote',
    'responsable': 'Steward Comercial',
    'causa': 'importe alterado a cero en una copia de prueba',
    'accion': 'restaurar desde la fuente ficticia preservada y revalidar',
    'autorizacion': 'simulada; no representa firma ni autenticacion',
}
fuente_recuperada = subconjunto.copy(deep=True)
incidente['estado'] = 'CORREGIDO_PENDIENTE_VALIDACION'
recuperacion = ejecutar_etl(fuente_recuperada, productos, referencia, conexion)
assert recuperacion['estado'] == 'PUBLICADO'
assert recuperacion['nuevas'] == 0 and recuperacion['existentes'] == 4
assert conexion.execute('SELECT COUNT(*) FROM ventas').fetchone()[0] == 4
incidente['lote_recuperacion'] = recuperacion['lote']
incidente['estado'] = 'CERRADO_VERIFICADO'
print(json_seguro(incidente))
display(pd.read_sql_query('SELECT estado, total, rechazadas, nuevas, existentes FROM lotes', conexion))
conexion.close()
print('Recuperacion verificada sin relajar umbrales ni duplicar ventas.')

## Resultados esperados y mejora continua

El tablero debe mostrar PUBLICADO, BLOQUEADO y BLOQUEADO. La deriva de esquema genera un evento separado de preflight. El reintento del subconjunto recuperado queda publicado sin ventas nuevas, y el incidente termina CERRADO_VERIFICADO con referencias al lote fallido y al de recuperacion.

En un proceso real el cierre requiere comprobar causa raiz, restauracion del servicio, calidad del dato, consumidores afectados y aprobacion. Un lote publicado no implica que desaparecieron incidentes previos ni las advertencias del caso.

### Ejercicios

1. Incorporar un lote vacio y un control de volumen esperado. Registrar por que no debe publicarse aunque no haya filas que fallen.
2. Implementar una bitacora persistente de incidentes con transiciones permitidas y evidencia de responsable, fecha y decision. No confundirla con autenticacion.
3. Anadir alertas por cambio de volumen y por cobertura de referencia baja. Evaluar tendencia solamente entre poblaciones comparables.
4. Disenar metricas operativas: latencia, tasa de reintentos, tiempo de resolucion y recurrencia. Definir SLI y SLO antes de hablar de SLA.

**Limites:** no se envian notificaciones reales ni se verifica una aprobacion de identidad. El tablero usa una poblacion minima y artificial; no sirve para elegir umbrales de produccion. El preflight y los incidentes solo viven en memoria en esta demo; su persistencia es un requisito de adopcion.